## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [19]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Dirichlet)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  inputs_left = tf.concat([x_left, t], axis=1)
  inputs_right = tf.concat([x_right, t], axis=1)
  p_left = model(inputs_left)
  p_right = model(inputs_right)

  return tf.reduce_mean(
      tf.square(p_left)
  ) + tf.reduce_mean(
      tf.square(p_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.sin(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_dirichlet = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_dirichlet = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_dirichlet, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_dirichlet.numpy())
    loss_history_dirichlet.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_dirichlet.numpy()}")

Epoch 0, Loss: 0.6452118158340454
Epoch 50, Loss: 0.24360010027885437
Epoch 100, Loss: 0.17938390374183655
Epoch 150, Loss: 0.12929274141788483
Epoch 200, Loss: 0.08007371425628662
Epoch 250, Loss: 0.05576062947511673
Epoch 300, Loss: 0.03833121806383133
Epoch 350, Loss: 0.030464230105280876
Epoch 400, Loss: 0.020356858149170876
Epoch 450, Loss: 0.016298852860927582
Epoch 500, Loss: 0.01167009212076664
Epoch 550, Loss: 0.00995673704892397
Epoch 600, Loss: 0.008860688656568527
Epoch 650, Loss: 0.0060906438156962395
Epoch 700, Loss: 0.006414498668164015
Epoch 750, Loss: 0.0038990783505141735
Epoch 800, Loss: 0.005090104416012764
Epoch 850, Loss: 0.004490295425057411
Epoch 900, Loss: 0.0034147033002227545
Epoch 950, Loss: 0.0030459719710052013


In [20]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.2. Piezoconductivity Equation (Neumann)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos(x)$, boundary conditions $p_x(0,t) = 0$, $p_x(\pi,t) = 0$.

In [21]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  return tf.reduce_mean(
      tf.square(p_x_left)
  ) + tf.reduce_mean(
      tf.square(p_x_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_neumann = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_neumann = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_neumann, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_neumann.numpy())
    loss_history_neumann.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_neumann.numpy()}")

Epoch 0, Loss: 2.942626953125
Epoch 50, Loss: 0.43496108055114746
Epoch 100, Loss: 0.3367888927459717
Epoch 150, Loss: 0.27930229902267456
Epoch 200, Loss: 0.23980262875556946
Epoch 250, Loss: 0.20052358508110046
Epoch 300, Loss: 0.1736617237329483
Epoch 350, Loss: 0.13460402190685272
Epoch 400, Loss: 0.11887755990028381
Epoch 450, Loss: 0.08924595266580582
Epoch 500, Loss: 0.08653444796800613
Epoch 550, Loss: 0.07800757884979248
Epoch 600, Loss: 0.05837910622358322
Epoch 650, Loss: 0.058223478496074677
Epoch 700, Loss: 0.05004676431417465
Epoch 750, Loss: 0.042720142751932144
Epoch 800, Loss: 0.040432680398225784
Epoch 850, Loss: 0.028535686433315277
Epoch 900, Loss: 0.02630903758108616
Epoch 950, Loss: 0.026411864906549454


In [22]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.3. Piezoconductivity Equation (Robin)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos\left(x - \frac{\pi}{4}\right)$, boundary conditions $p_x(0,t) - p(0,t) = 0$, $p_x(\pi,t) - p(\pi,t) = 0$.

In [23]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Robin)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  loss_left = tf.reduce_mean(tf.square(p_x_left - p_left))
  loss_right = tf.reduce_mean(tf.square(p_x_right - p_right))

  return loss_left + loss_right

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x - np.pi / 4)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_robin = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_robin = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_robin, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_robin.numpy())
    loss_history_robin.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_robin.numpy()}")

Epoch 0, Loss: 1.6873172521591187
Epoch 50, Loss: 0.40414050221443176
Epoch 100, Loss: 0.3640018701553345
Epoch 150, Loss: 0.24070772528648376
Epoch 200, Loss: 0.16436955332756042
Epoch 250, Loss: 0.10501723736524582
Epoch 300, Loss: 0.0751953125
Epoch 350, Loss: 0.05425282567739487
Epoch 400, Loss: 0.02808617427945137
Epoch 450, Loss: 0.027633659541606903
Epoch 500, Loss: 0.01994303986430168
Epoch 550, Loss: 0.01483607105910778
Epoch 600, Loss: 0.011698641814291477
Epoch 650, Loss: 0.010885444469749928
Epoch 700, Loss: 0.008319384418427944
Epoch 750, Loss: 0.007138634100556374
Epoch 800, Loss: 0.007723306305706501
Epoch 850, Loss: 0.005834808107465506
Epoch 900, Loss: 0.005465186201035976
Epoch 950, Loss: 0.0056837270967662334


In [24]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot - np.pi / 4)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## LOSS:

In [26]:
import plotly.graph_objects as go

epochs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
          550, 600, 650, 700, 750, 800, 850, 900, 950]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_history_dirichlet, mode='lines+markers', name='Дирихле'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_neumann, mode='lines+markers', name='Нейман'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_robin, mode='lines+markers', name='Робин'))
fig.update_layout(
    title='Сходимость обучения PINN',
    xaxis_title='Эпоха',
    yaxis_title='Функция потерь',
    yaxis_type='log',  # логарифмическая шкала
    template='plotly_white'
)
fig.show()